# ConnectWise
### Voice-first assistant for people with visual disabilities
Built with ipywidgets | Groq | YouTube | Africa's Talking


## Step 1 - Install Dependencies

In [ ]:
!pip install ipywidgets africastalking requests groq PyPDF2 -q
print('All dependencies installed')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.7/139.7 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 26.2 MB/s eta 0:00:00
All dependencies installed


## Step 2 - Configuration

Add api keys on secret keys

- Africa Talking Username
- Africa's Talking API key
- YouTube Data API v3 key
- Groq API key




In [ ]:
from google.colab import userdata


CLOUD_RUN_URL = userdata.get('CLOUD_RUN_URL')

AT_USERNAME = 'sandbox'
AT_API_KEY  = userdata.get('AT_API')
YOUTUBE_KEY = userdata.get('YOUTUBE_KEY')

print('Configuration loaded successfully')
print('Cloud Run URL :', CLOUD_RUN_URL)
print('AT Username   :', AT_USERNAME)
print('AT API Key    :', AT_API_KEY[:8], '...')

Configuration loaded successfully
Cloud Run URL : https://voiceai-api-x4rb6gzrxq-uc.a.run.app
AT Username   : sandbox
AT API Key    : atsk_cab ...


## Step 3 - Backend Functions

In [ ]:
import requests
import base64
import africastalking
import re
import json as _json
from groq import Groq
from google.colab import userdata

# Contacts store
contacts = {}

def add_contact(name, phone):
    clean_name = re.sub(r'[^\w\s]', '', name.lower().strip())
    contacts[clean_name] = phone.strip()

def find_contact(name):
    """Contact lookup: """
    # 1. Clean: lowercase, strip whitespace and punctuation
    cleaned = re.sub(r'[^\w\s]', '', name.lower().strip())

    # 2. Exact match after cleaning
    if cleaned in contacts:
        return contacts[cleaned]

    # 3. Partial match — check if any contact name appears anywhere in the spoken phrase
    for contact_name, phone in contacts.items():
        if contact_name in cleaned:
            return phone

    # 4. Fuzzy match — handle speech misrecognition (e.g. "jon" vs "john")
    #Uses SequenceMatcher for similarity scoring
    from difflib import SequenceMatcher
    best_score = 0
    best_phone = None
    for contact_name, phone in contacts.items():
        score = SequenceMatcher(None, cleaned, contact_name).ratio()
        if score > best_score:
            best_score = score
            best_phone = phone

    # Only accept if similarity is above 80%
    if best_score >= 0.80:
        return best_phone

    return None


# Frustration detector - Groq api
def analyse_frustration(text):
    try:
        groq_key = userdata.get('GROQ_KEY')
        print(f'[DEBUG] GROQ_KEY loaded: {groq_key[:8]}...' if groq_key else '[DEBUG] GROQ_KEY is EMPTY/NONE')
        client = Groq(api_key=groq_key)
        prompt = (
            "Analyse the emotional tone of this message and reply with ONLY valid JSON, "
            "no markdown fences, no extra text. Keys required:\n"
            "  frustration_score: integer 0-100 (0=calm, 100=extremely frustrated)\n"
            "  emotion: one word — calm | mild | frustrated | very_frustrated\n"
            "  suggested_response: a short empathetic reply under 25 words\n\n"
            f'Message: "{text}"'
        )
        print(f'[DEBUG] Calling Groq with model llama3-8b-8192...')
        chat = client.chat.completions.create(
            model='llama-3.3-70b-versatile',
            messages=[{'role': 'user', 'content': prompt}],
            temperature=0.2,
            max_tokens=150,
        )
        raw = chat.choices[0].message.content.strip()
        print(f'[DEBUG] Groq raw response: {raw}')
        raw = raw.replace('```json', '').replace('```', '').strip()
        data = _json.loads(raw)
        data['frustration_score'] = max(0, min(100, int(data.get('frustration_score', 0))))
        print(f'[DEBUG] Parsed: score={data["frustration_score"]}, emotion={data["emotion"]}')
        return data
    except Exception as e:
        print(f'[DEBUG] analyse_frustration FAILED: {type(e).__name__}: {e}')
        return {'frustration_score': 0, 'emotion': 'calm',
                'suggested_response': 'How can I help you today?'}


# YouTube search — direct YouTube Data API
def search_youtube(query):
    try:
        yt_key = userdata.get('YOUTUBE_KEY')
        print(f'[DEBUG] YOUTUBE_KEY loaded: {yt_key[:8]}...' if yt_key else '[DEBUG] YOUTUBE_KEY is EMPTY/NONE')
        r = requests.get(
            'https://www.googleapis.com/youtube/v3/search',
            params={
                'part': 'snippet',
                'q': query,
                'type': 'video',
                'maxResults': 3,
                'key': yt_key,
            },
            timeout=10
        )
        print(f'[DEBUG] YouTube API status: {r.status_code}')
        r.raise_for_status()
        items = r.json().get('items', [])
        print(f'[DEBUG] YouTube results count: {len(items)}')
        results = [
            {
                'video_id': item['id']['videoId'],
                'title':    item['snippet']['title'],
                'channel':  item['snippet']['channelTitle'],
            }
            for item in items
        ]
        return {'results': results}
    except Exception as e:
        print(f'[DEBUG] search_youtube FAILED: {type(e).__name__}: {e}')
        return {'results': []}


# PDF summarise — direct Groq call (no Cloud Run)
def summarise_pdf(pdf_bytes):
    try:
        import PyPDF2
        import io
        reader = PyPDF2.PdfReader(io.BytesIO(pdf_bytes))
        page_count = len(reader.pages)
        # Extract text from all pages (cap at 6000 chars to stay within token limit)
        full_text = ' '.join(
            page.extract_text() or '' for page in reader.pages
        )[:6000].strip()
        if not full_text:
            return {'summary': 'Could not extract text from this PDF (it may be scanned/image-based).', 'page_count': page_count}
        client = Groq(api_key=userdata.get('GROQ_KEY')) # Corrected GROK_KEY to GROQ_KEY
        chat = client.chat.completions.create(
            model='llama-3.3-70b-versatile',
            messages=[{
                'role': 'user',
                'content': (
                    f'Summarise the following document in 3-5 clear sentences. '
                    f'Be concise and focus on the key points.\n\nDocument:\n{full_text}'
                )
            }],
            temperature=0.3,
            max_tokens=300,
        )
        summary = chat.choices[0].message.content.strip()
        return {'summary': summary, 'page_count': page_count}
    except Exception as e:
        print(f'[DEBUG] summarise_pdf FAILED: {type(e).__name__}: {e}')
        return {'summary': f'Error reading PDF: {e}', 'page_count': 0}


# SMS function
def send_sms(phone_number, message):
    try:
        africastalking.initialize(AT_USERNAME, AT_API_KEY)
        sms      = africastalking.SMS
        response = sms.send(message=message, recipients=[phone_number])
        return {'status': 'success', 'response': str(response)}
    except Exception as e:
        return {'status': 'error', 'message': str(e)}


# Airtime function
def buy_airtime(phone_number, amount, currency='ZAR'):
    try:
        africastalking.initialize(AT_USERNAME, AT_API_KEY)
        airtime  = africastalking.Airtime
        response = airtime.send(
            phone_number=phone_number,
            amount=float(amount),
            currency_code=currency
        )
        return {'status': 'success', 'response': str(response)}
    except Exception as e:
        return {'status': 'error', 'message': str(e)}


print('✅ Backend functions ready (debug mode ON)')
print('✅ Contacts store ready')

✅ Backend functions ready (debug mode ON)
✅ Contacts store ready


## Step 4 - Launch ConnectWise App

In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML, Javascript, clear_output
import builtins

display(HTML('''
<style>
@import url("https://fonts.googleapis.com/css2?family=Poppins:wght@400;600;700;800&display=swap");
.cw-app {
  font-family: Poppins, sans-serif;
  background: linear-gradient(160deg, #0d1b2a 0%, #112240 60%, #0d1b2a 100%);
  border-radius: 20px; overflow: hidden; color: white;
  max-width: 520px; margin: 0 auto;
  box-shadow: 0 20px 60px rgba(0,0,0,0.5);
}
.cw-header {
  background: linear-gradient(90deg, #1565c0 0%, #1976d2 100%);
  padding: 20px 24px; display: flex; align-items: center; gap: 14px;
  box-shadow: 0 4px 20px rgba(21,101,192,0.5);
}
.cw-logo {
  width: 48px; height: 48px; background: rgba(255,255,255,0.15);
  border-radius: 14px; display: flex; align-items: center;
  justify-content: center; font-size: 24px;
}
.cw-title { font-size: 1.7rem; font-weight: 800; letter-spacing: -0.5px; }
.cw-subtitle { font-size: 0.72rem; opacity: 0.75; }
.cw-card {
  background: rgba(255,255,255,0.05); border: 1px solid rgba(255,255,255,0.1);
  border-radius: 14px; padding: 16px 18px; width: 100%;
}
.cw-card-label {
  font-size: 0.62rem; text-transform: uppercase;
  letter-spacing: 0.1em; opacity: 0.45; margin-bottom: 7px;
}
.cw-card-text { font-size: 0.92rem; line-height: 1.65; opacity: 0.9; }
.cw-frust-row { display: flex; align-items: center; gap: 12px; width: 100%; }
.cw-frust-bar-wrap { flex: 1; }
.cw-frust-lbl {
  font-size: 0.62rem; text-transform: uppercase;
  letter-spacing: 0.1em; opacity: 0.45; margin-bottom: 6px;
}
.cw-frust-track { height: 6px; background: rgba(255,255,255,0.1); border-radius: 100px; overflow: hidden; }
.cw-frust-fill { height: 100%; border-radius: 100px; background: #4caf50; transition: width 0.6s ease, background 0.3s; }
.cw-panel {
  background: rgba(255,255,255,0.05); border: 1px solid rgba(255,255,255,0.1);
  border-radius: 14px; padding: 20px; width: 100%;
  display: flex; flex-direction: column; gap: 10px;
}
.cw-panel-title { font-size: 0.95rem; font-weight: 700; margin-bottom: 4px; }
.cw-field-label { font-size: 0.75rem; opacity: 0.55; margin-bottom: 2px; }
.cw-video-wrap { width: 100%; border-radius: 12px; overflow: hidden; aspect-ratio: 16/9; background: #000; }
.cw-video-wrap iframe { width: 100%; height: 100%; border: none; }
.cw-status {
  font-size: 0.8rem; padding: 9px 14px; border-radius: 10px;
  background: rgba(21,101,192,0.18); border: 1px solid rgba(21,101,192,0.3);
  color: #90caf9; width: 100%;
}
.cw-status.success { background: rgba(76,175,80,0.18); border-color: rgba(76,175,80,0.3); color: #a5d6a7; }
.cw-status.error   { background: rgba(244,67,54,0.18); border-color: rgba(244,67,54,0.3); color: #ef9a9a; }
.cw-divider { height: 1px; background: rgba(255,255,255,0.08); margin: 4px 0; }
.widget-text input, .widget-textarea textarea {
  background: rgba(255,255,255,0.08) !important;
  border: 1px solid rgba(255,255,255,0.15) !important;
  border-radius: 10px !important; color: white !important;
  font-family: Poppins, sans-serif !important;
}
.widget-button button {
  background: linear-gradient(90deg, #1565c0, #1976d2) !important;
  border: none !important; border-radius: 10px !important;
  color: white !important; font-family: Poppins, sans-serif !important;
  font-weight: 700 !important;
}
</style>
'''))


out_status      = widgets.Output()
out_frustration = widgets.Output()
out_transcript  = widgets.Output()
out_response    = widgets.Output()
out_panel       = widgets.Output()


def show_status(msg, kind='info'):
    with out_status:
        clear_output()
        display(HTML(f'<div class="cw-status {kind}">{msg}</div>'))


def show_frustration(score, emotion):
    color = '#4caf50' if score < 40 else '#ff9800' if score < 65 else '#f44336'
    emoji = '😌' if score < 21 else '😐' if score < 41 else '😤' if score < 66 else '😡'
    with out_frustration:
        clear_output()
        display(HTML(f'''
        <div class="cw-frust-row">
          <div style="font-size:1.8rem">{emoji}</div>
          <div class="cw-frust-bar-wrap">
            <div class="cw-frust-lbl">Frustration Level</div>
            <div class="cw-frust-track">
              <div class="cw-frust-fill" style="width:{score}%;background:{color}"></div>
            </div>
          </div>
          <div style="font-size:0.85rem;font-weight:700;color:{color};min-width:36px;text-align:right">{score}%</div>
        </div>
        '''))


def speak(text):
    safe = text.replace('"', '').replace("'", '')[:300]
    display(Javascript(f'''
    window.speechSynthesis.cancel();
    const u = new SpeechSynthesisUtterance("{safe}");
    u.lang = "en-ZA"; u.rate = 0.95;
    window.speechSynthesis.speak(u);
    '''))


def handle_voice_input(transcript):
    with out_transcript:
        clear_output()
        display(HTML(f'''
        <div class="cw-card">
          <div class="cw-card-label">You said</div>
          <div class="cw-card-text">{transcript}</div>
        </div>
        '''))

    show_status('Analysing...', 'info')
    result  = analyse_frustration(transcript)
    score   = result.get('frustration_score', 0)
    emotion = result.get('emotion', 'calm')
    reply   = result.get('suggested_response', 'How can I help you?')

    show_frustration(score, emotion)

    cmd = transcript.lower()
    if 'play' in cmd:
        query = cmd.replace('play', '').strip()
        show_status('Searching YouTube for: ' + query, 'info')
        yt = search_youtube(query)
        if yt.get('results'):
            video = yt['results'][0]
            reply = 'Now playing: ' + video['title']
            with out_panel:
                clear_output()
                display(HTML(f'''
                <div class="cw-panel">
                  <div class="cw-panel-title">Now Playing</div>
                  <div class="cw-video-wrap">
                    <iframe src="https://www.youtube.com/embed/{video["video_id"]}?autoplay=1"
                      allowfullscreen allow="autoplay; encrypted-media"></iframe>
                  </div>
                  <div style="font-size:0.82rem;opacity:0.6;margin-top:4px">{video['title']}</div>
                </div>
                '''))

    with out_response:
        clear_output()
        display(HTML(f'''
        <div class="cw-card">
          <div class="cw-card-label">ConnectWise says</div>
          <div class="cw-card-text">{reply}</div>
        </div>
        '''))

    show_status('Done - tap mic to speak again', 'success')
    speak(reply)
    if hasattr(builtins, 'reset_mic'):
        builtins.reset_mic()


builtins.handle_voice_input = handle_voice_input
builtins.show_status        = show_status


# Airtime Panel
def show_airtime_panel(btn):
    with out_panel:
        clear_output()

        airtime_state = {'step': 1, 'phone': '', 'name': '', 'amount': '', 'currency': 'ZAR'}

        title      = widgets.HTML('<div class="cw-panel-title">📱 Buy Airtime</div>')
        info       = widgets.HTML('')
        at_input   = widgets.Text(
            placeholder='Type here or use mic below...',
            layout=widgets.Layout(width='100%')
        )
        submit     = widgets.Button(description='Next',
                        layout=widgets.Layout(width='100%'))
        submit.style.button_color = '#1565c0'
        result_out = widgets.Output()

        # Currency dropdown
        lbl_curr = widgets.HTML('<div class="cw-field-label">Currency</div>')
        currency = widgets.Dropdown(
            options=['ZAR', 'KES', 'UGX', 'TZS', 'NGN', 'GHS'],
            value='ZAR', layout=widgets.Layout(width='100%')
        )
        lbl_curr.layout.display = 'none'
        currency.layout.display = 'none'

        # Airtime mic button
        at_mic = widgets.Button(
            description='🎤  Speak',
            layout=widgets.Layout(width='120px', height='80px')
        )
        at_mic.style.button_color = '#1565c0'
        at_mic.style.font_weight  = 'bold'
        at_mic_out = widgets.Output()

        def set_step1():
            airtime_state['step'] = 1
            lbl_curr.layout.display = 'none'
            currency.layout.display = 'none'
            info.value = '''
            <div style="color:white;font-size:0.9rem;padding:8px 0;
              font-family:Poppins;line-height:1.6">
              <strong>Step 1:</strong> Who are you buying airtime for?<br/>
              <span style="opacity:0.6;font-size:0.78rem">
              Say or type a contact name or phone number
              </span>
            </div>
            '''
            at_input.placeholder = 'Contact name or +27XXXXXXXXX'
            at_input.value       = ''
            submit.description   = 'Next'
            speak('Who are you buying airtime for? Say a contact name or phone number.')

        def set_step2():
            airtime_state['step'] = 2
            lbl_curr.layout.display = ''
            currency.layout.display = ''
            name_or_phone = airtime_state['name'] or airtime_state['phone']
            info.value = f'''
            <div style="color:white;font-size:0.9rem;padding:8px 0;
              font-family:Poppins;line-height:1.6">
              <strong>Step 2:</strong> Buying for <strong>{name_or_phone}</strong><br/>
              <span style="opacity:0.6;font-size:0.78rem">
              How much airtime? (e.g. say "10" or "50")
              </span>
            </div>
            '''
            at_input.value       = ''
            at_input.placeholder = 'Amount e.g. 10'
            submit.description   = 'Buy Airtime'
            speak(f'How much airtime do you want to send to {name_or_phone}?')

        def process_at_input(val):
            val = val.strip()
            if not val:
                return

            if airtime_state['step'] == 1:
                # Try contacts first
                phone = find_contact(val)
                if phone:
                    airtime_state['phone'] = phone
                    airtime_state['name']  = val.title()
                    with result_out:
                        clear_output()
                        display(HTML(f'<div class="cw-status success">Found {val.title()} — {phone}</div>'))
                    set_step2()
                else:
                    # Try raw phone number
                    match = re.search(r'\+?\d[\d\s\-]{8,14}', val)
                    if match:
                        phone = match.group(0).replace(' ', '').replace('-', '')
                        if not phone.startswith('+'):
                            phone = '+' + phone
                        airtime_state['phone'] = phone
                        airtime_state['name']  = phone
                        set_step2()
                    else:
                        with result_out:
                            clear_output()
                            display(HTML(f'''
                            <div class="cw-status error">
                              Contact "{val}" not found. Add them in Contacts first
                              or say their number directly.
                            </div>
                            '''))
                        speak(f'I could not find {val} in your contacts. Please add them first or say their number.')

            elif airtime_state['step'] == 2:
                # Extract numeric/number amount from user
                word_map = {
                    'one':'1','two':'2','three':'3','four':'4','five':'5',
                    'six':'6','seven':'7','eight':'8','nine':'9','ten':'10',
                    'twenty':'20','thirty':'30','fifty':'50','hundred':'100'
                }
                clean = val.lower()
                for word, num in word_map.items():
                    clean = clean.replace(word, num)
                num_match = re.search(r'\d+\.?\d*', clean)
                if not num_match:
                    with result_out:
                        clear_output()
                        display(HTML('<div class="cw-status error">Please say a valid amount e.g. 10 or 50</div>'))
                    speak('Please say a valid amount like 10 or 50.')
                    return
                airtime_state['amount']   = num_match.group(0)
                airtime_state['currency'] = currency.value
                name_or_phone = airtime_state['name'] or airtime_state['phone']
                with result_out:
                    clear_output()
                    display(HTML('<div class="cw-status">Sending airtime...</div>'))
                res = buy_airtime(airtime_state['phone'], airtime_state['amount'], airtime_state['currency'])
                with result_out:
                    clear_output()
                    if res['status'] == 'success':
                        msg = f"Airtime of {airtime_state['currency']} {airtime_state['amount']} sent to {name_or_phone} successfully!"
                        display(HTML(f'''
                        <div class="cw-status success">✅ Airtime sent!</div>
                        <div class="cw-card" style="margin-top:8px">
                          <div class="cw-card-label">Airtime Details</div>
                          <div class="cw-card-text">
                            To: {name_or_phone} ({airtime_state['phone']})<br/>
                            Amount: {airtime_state['currency']} {airtime_state['amount']}<br/>
                            <span style="opacity:0.6;font-size:0.8rem">{res['response']}</span>
                          </div>
                        </div>
                        '''))
                        speak(msg)
                        # Reset
                        airtime_state['step']   = 1
                        airtime_state['phone']  = ''
                        airtime_state['name']   = ''
                        airtime_state['amount'] = ''
                        at_input.value = ''
                        set_step1()
                    else:
                        display(HTML(f'<div class="cw-status error">Error: {res["message"]}</div>'))
                        speak('Sorry, could not send airtime.')

        def on_at_submit(b):
            process_at_input(at_input.value)
            at_input.value = ''

        submit.on_click(on_at_submit)

        # Airtime mic click
        def on_at_mic(b):
            at_mic.description        = '🔴  Listening...'
            at_mic.style.button_color = '#c62828'
            with at_mic_out:
                clear_output()
                display(HTML('<div style="color:white;font-size:0.8rem;font-family:Poppins">🎤 Listening...</div>'))
            display(Javascript('''
            (() => {
              const SR = window.SpeechRecognition || window.webkitSpeechRecognition;
              if (!SR) { alert("Use Chrome browser"); return; }
              navigator.mediaDevices.getUserMedia({ audio: true })
                .then(stream => {
                  stream.getTracks().forEach(t => t.stop());
                  let tries = 0;
                  function go() {
                    if (tries >= 3) return;
                    tries++;
                    const r = new SR();
                    r.lang = "en-ZA"; r.continuous = false; r.interimResults = false;
                    r.onresult = (e) => {
                      const t = e.results[0][0].transcript;
                      console.log("Airtime transcript:", t);
                      google.colab.kernel.invokeFunction("notebook.airtime_input", [t], {});
                    };
                    r.onerror = (e) => {
                      if (e.error === "network" || e.error === "aborted") setTimeout(go, 800);
                      else google.colab.kernel.invokeFunction("notebook.airtime_input", [""], {});
                    };
                    r.onend = () => console.log("Airtime listening ended");
                    r.start();
                  }
                  go();
                })
                .catch(() => alert("Microphone access denied"));
            })();
            '''))

        at_mic.on_click(on_at_mic)

        from google.colab import output as colab_output
        def at_callback(val):
            val = str(val).strip()
            at_mic.description        = '🎤  Speak'
            at_mic.style.button_color = '#1565c0'
            with at_mic_out:
                clear_output()
            if val:
                process_at_input(val)
            return val
        colab_output.register_callback('notebook.airtime_input', at_callback)

        box = widgets.VBox([
            title, info,
            widgets.HTML('<div class="cw-field-label">Speak or type:</div>'),
            at_mic, at_mic_out,
            at_input, lbl_curr, currency, submit, result_out
        ], layout=widgets.Layout(width='100%'))
        display(box)
        set_step1()



# SMS Panel
def show_sms_panel(btn):
    with out_panel:
        clear_output()

        sms_state = {'step': 1, 'phone': '', 'name': '', 'message': ''}

        title      = widgets.HTML('<div class="cw-panel-title">💬 Send SMS</div>')
        info       = widgets.HTML('')
        sms_input  = widgets.Text(
            placeholder='Type here or use mic below...',
            layout=widgets.Layout(width='100%')
        )
        submit     = widgets.Button(description='Next',
                        layout=widgets.Layout(width='100%'))
        submit.style.button_color = '#1565c0'
        result_out = widgets.Output()

        # SMS mic button
        sms_mic = widgets.Button(
            description='🎤  Speak',
            layout=widgets.Layout(width='120px', height='80px')
        )
        sms_mic.style.button_color = '#1565c0'
        sms_mic.style.font_weight  = 'bold'

        sms_mic_out = widgets.Output()

        def set_step1():
            sms_state['step'] = 1
            info.value = '''
            <div style="color:white;font-size:0.9rem;padding:8px 0;
              font-family:Poppins;line-height:1.6">
              <strong>Step 1:</strong> Who do you want to SMS?<br/>
              <span style="opacity:0.6;font-size:0.78rem">
              Say or type a contact name or phone number
              </span>
            </div>
            '''
            sms_input.placeholder = 'Contact name or +27XXXXXXXXX'
            submit.description    = 'Next'
            speak('Who do you want to SMS? Say a contact name or phone number.')

        def set_step2():
            sms_state['step'] = 2
            name_or_phone = sms_state['name'] or sms_state['phone']
            info.value = f'''
            <div style="color:white;font-size:0.9rem;padding:8px 0;
              font-family:Poppins;line-height:1.6">
              <strong>Step 2:</strong> Sending to <strong>{name_or_phone}</strong><br/>
              <span style="opacity:0.6;font-size:0.78rem">
              What is your message?
              </span>
            </div>
            '''
            sms_input.value       = ''
            sms_input.placeholder = 'Type your message...'
            submit.description    = 'Send SMS'
            speak(f'What is your message to {name_or_phone}?')

        def process_input(val):
            val = val.strip()
            if not val:
                return

            if sms_state['step'] == 1:
                # Try contacts first
                phone = find_contact(val)
                if phone:
                    sms_state['phone'] = phone
                    sms_state['name']  = val.title()
                    with result_out:
                        clear_output()
                        display(HTML(f'<div class="cw-status success">Found {val.title()} - {phone}</div>'))
                    set_step2()
                else:
                    # Try extract phone number
                    match = re.search(r'(?i)\+?\d[\d\s\-A-Z]{8,20}', val)
                    if match:
                        phone = match.group(0).replace(' ', '').replace('-', '')
                        if not phone.startswith('+'):
                            phone = '+' + phone
                        sms_state['phone'] = phone
                        sms_state['name']  = phone
                        set_step2()
                    else:
                        with result_out:
                            clear_output()
                            display(HTML(f'''
                            <div class="cw-status error">
                              Contact "{val}" not found. Add them in Contacts first
                              or say their number directly.
                            </div>
                            '''))
                        speak(f'I could not find {val} in your contacts. Please add them first.')

            elif sms_state['step'] == 2:
                sms_state['message'] = val
                with result_out:
                    clear_output()
                    display(HTML('<div class="cw-status">Sending SMS...</div>'))
                res = send_sms(sms_state['phone'], sms_state['message'])
                with result_out:
                    clear_output()
                    if res['status'] == 'success':
                        name_or_phone = sms_state['name'] or sms_state['phone']
                        display(HTML(f'''
                        <div class="cw-status success">SMS sent successfully!</div>
                        <div class="cw-card" style="margin-top:8px">
                          <div class="cw-card-label">SMS Details</div>
                          <div class="cw-card-text">
                            To: {name_or_phone} ({sms_state["phone"]})<br/>
                            Message: {sms_state["message"]}
                          </div>
                        </div>
                        '''))
                        speak(f'SMS sent to {name_or_phone} successfully')
                        # Reset
                        sms_state['step']    = 1
                        sms_state['phone']   = ''
                        sms_state['name']    = ''
                        sms_state['message'] = ''
                        sms_input.value = ''
                        set_step1()
                    else:
                        display(HTML(f'<div class="cw-status error">Error: {res["message"]}</div>'))
                        speak('Sorry, could not send SMS.')

        def on_submit(b):
            process_input(sms_input.value)
            sms_input.value = ''

        submit.on_click(on_submit)

        # SMS mic
        def on_sms_mic(b):
            sms_mic.description        = '🔴  Listening...'
            sms_mic.style.button_color = '#c62828'
            with sms_mic_out:
                clear_output()
                display(HTML('<div style="color:white;font-size:0.8rem;font-family:Poppins">🎤 Listening...</div>'))
            display(Javascript('''
            (() => {
              const SR = window.SpeechRecognition || window.webkitSpeechRecognition;
              if (!SR) { alert("Use Chrome browser"); return; }
              navigator.mediaDevices.getUserMedia({ audio: true })
                .then(stream => {
                  stream.getTracks().forEach(t => t.stop());
                  let tries = 0;
                  function go() {
                    if (tries >= 3) return;
                    tries++;
                    const r = new SR();
                    r.lang = "en-ZA"; r.continuous = false; r.interimResults = false;
                    r.onresult = (e) => {
                      const t = e.results[0][0].transcript;
                      console.log("SMS transcript:", t);
                      google.colab.kernel.invokeFunction("notebook.sms_input", [t], {});
                    };
                    r.onerror = (e) => {
                      if (e.error === "network" || e.error === "aborted") setTimeout(go, 800);
                      else google.colab.kernel.invokeFunction("notebook.sms_input", [""], {});
                    };
                    r.onend = () => console.log("SMS listening ended");
                    r.start();
                  }
                  go();
                })
                .catch(() => alert("Microphone access denied"));
            })();
            '''))

        sms_mic.on_click(on_sms_mic)

        from google.colab import output as colab_output
        def sms_callback(val):
            val = str(val).strip()
            sms_mic.description        = '🎤  Speak'
            sms_mic.style.button_color = '#1565c0'
            with sms_mic_out:
                clear_output()
            if val:
                process_input(val)
            return val
        colab_output.register_callback('notebook.sms_input', sms_callback)

        box = widgets.VBox([
            title, info,
            widgets.HTML('<div class="cw-field-label">Speak or type:</div>'),
            sms_mic, sms_mic_out,
            sms_input, submit, result_out
        ], layout=widgets.Layout(width='100%'))
        display(box)
        set_step1()


# PDF Panel
def show_pdf_panel(btn):
    with out_panel:
        clear_output()
        title      = widgets.HTML('<div class="cw-panel-title">📄 Summarise PDF</div>')
        lbl        = widgets.HTML('<div class="cw-field-label">Upload your PDF file</div>')
        upload     = widgets.FileUpload(accept='.pdf', multiple=False,
                        layout=widgets.Layout(width='100%'))
        submit     = widgets.Button(description='Summarise PDF',
                        layout=widgets.Layout(width='100%'))
        submit.style.button_color = '#1565c0'
        result_out = widgets.Output()

        def do_pdf(b):
            with result_out:
                clear_output()
                if not upload.value:
                    display(HTML('<div class="cw-status error">Please upload a PDF first</div>'))
                    return
                display(HTML('<div class="cw-status">Reading PDF...</div>'))
            pdf_bytes = list(upload.value.values())[0]['content']
            result    = summarise_pdf(bytes(pdf_bytes))
            summary   = result.get('summary', 'Could not read PDF')
            pages     = result.get('page_count', 0)
            with result_out:
                clear_output()
                display(HTML(f'''
                <div class="cw-card">
                  <div class="cw-card-label">PDF Summary - {pages} pages</div>
                  <div class="cw-card-text">{summary}</div>
                </div>
                '''))
            speak(summary)
        submit.on_click(do_pdf)

        box = widgets.VBox([title, lbl, upload, submit, result_out],
                           layout=widgets.Layout(width='100%'))
        display(box)
        speak('Summarise PDF. Upload a PDF file and tap Summarise.')


# Contacts Panel
def show_contacts_panel(btn):
    with out_panel:
        clear_output()
        title      = widgets.HTML('<div class="cw-panel-title">👤 Contacts</div>')
        lbl_name   = widgets.HTML('<div class="cw-field-label">Name</div>')
        name_input = widgets.Text(placeholder='e.g. John',
                        layout=widgets.Layout(width='100%'))
        lbl_phone  = widgets.HTML('<div class="cw-field-label">Phone Number</div>')
        phone_input = widgets.Text(placeholder='e.g. +27831234567',
                         layout=widgets.Layout(width='100%'))
        save_btn   = widgets.Button(description='Save Contact',
                        layout=widgets.Layout(width='100%'))
        save_btn.style.button_color = '#1565c0'
        save_btn.style.font_weight  = 'bold'
        status_out = widgets.Output()
        list_out   = widgets.Output()

        def refresh_list():
            with list_out:
                clear_output()
                if contacts:
                    rows = ''.join([
                        f'<div style="display:flex;justify-content:space-between;'
                        f'padding:6px 0;border-bottom:1px solid rgba(255,255,255,0.08);'
                        f'font-family:Poppins">'
                        f'<span style="color:white;font-weight:600">{n.title()}</span>'
                        f'<span style="color:rgba(255,255,255,0.5);font-size:0.85rem">{p}</span>'
                        f'</div>'
                        for n, p in contacts.items()
                    ])
                    display(HTML(f'''
                    <div class="cw-card" style="margin-top:8px">
                      <div class="cw-card-label">Saved Contacts ({len(contacts)})</div>
                      {rows}
                    </div>
                    '''))
                else:
                    display(HTML(
                        '<div style="color:rgba(255,255,255,0.4);font-size:0.85rem;'
                        'padding:8px 0;font-family:Poppins">No contacts saved yet</div>'
                    ))

        def on_save(b):
            name  = name_input.value.strip()
            phone = phone_input.value.strip()
            if not name or not phone:
                with status_out:
                    clear_output()
                    display(HTML('<div class="cw-status error">Please enter both name and phone number</div>'))
                return
            add_contact(name, phone)
            name_input.value  = ''
            phone_input.value = ''
            with status_out:
                clear_output()
                display(HTML(f'<div class="cw-status success">{name.title()} saved!</div>'))
            speak(f'{name.title()} has been saved to your contacts')
            refresh_list()

        save_btn.on_click(on_save)

        box = widgets.VBox([
            title, lbl_name, name_input,
            lbl_phone, phone_input,
            save_btn, status_out, list_out
        ], layout=widgets.Layout(width='100%'))
        display(box)
        refresh_list()
        speak('Contacts. Add a name and phone number to get started.')


# App Header
display(HTML('''
<div class="cw-app">
  <div class="cw-header">
    <div class="cw-logo">🔵</div>
    <div>
      <div class="cw-title">ConnectWise</div>
      <div class="cw-subtitle">Voice-first accessibility assistant</div>
    </div>
  </div>
</div>
'''))

# Action Buttons
btn_airtime  = widgets.Button(description='📱  Airtime',
                layout=widgets.Layout(flex='1', height='60px'))
btn_sms      = widgets.Button(description='💬  SMS',
                layout=widgets.Layout(flex='1', height='60px'))
btn_pdf      = widgets.Button(description='📄  PDF',
                layout=widgets.Layout(flex='1', height='60px'))
btn_contacts = widgets.Button(description='👤  Contacts',
                layout=widgets.Layout(flex='1', height='60px'))

for b in [btn_airtime, btn_sms, btn_pdf, btn_contacts]:
    b.style.button_color = '#1565c0'
    b.style.font_weight  = 'bold'

btn_airtime.on_click(show_airtime_panel)
btn_sms.on_click(show_sms_panel)
btn_pdf.on_click(show_pdf_panel)
btn_contacts.on_click(show_contacts_panel)

display(HTML('<div style="font-family:Poppins,sans-serif;color:white;font-weight:700;margin:12px 0 8px 0">Actions:</div>'))
display(widgets.HBox(
    [btn_airtime, btn_sms, btn_pdf, btn_contacts],
    layout=widgets.Layout(width='100%')
))

display(out_status)
display(out_frustration)
display(out_transcript)
display(out_response)
display(out_panel)

show_status('ConnectWise is ready!', 'success')
print('Step 4 done. Run Step 5 for mic and Step 6 for text fallback.')

Output()

Output()

Output()

Output()

Output()

Step 4 done. Run Step 5 for mic and Step 6 for text fallback.


## Step 5 - Microphone

Must use **Chrome** browser. Allow microphone when prompted.


In [ ]:
from IPython.display import display, HTML, Javascript, clear_output
from google.colab import output as colab_output
import ipywidgets as widgets
import builtins

mic_out = widgets.Output()

mic_btn = widgets.Button(
    description='🎤  Tap to Speak',
    layout=widgets.Layout(width='180px', height='180px')
)
mic_btn.style.button_color = '#1565c0'
mic_btn.style.font_weight  = 'bold'


def reset_mic():
    mic_btn.description        = '🎤  Tap to Speak'
    mic_btn.style.button_color = '#1565c0'
    with mic_out:
        clear_output()
        display(HTML(
            '<div style="color:white;padding:8px;font-family:Poppins">'
            'Ready - tap to speak again</div>'
        ))


def set_transcript(val):
    val = str(val).strip()
    print('Received:', val)
    mic_btn.description        = '🎤  Tap to Speak'
    mic_btn.style.button_color = '#1565c0'
    with mic_out:
        clear_output()
    if val:
        handle_voice_input(val)
    return val


colab_output.register_callback('notebook.set_transcript', set_transcript)


def on_mic_click(b):
    mic_btn.description        = '🔴  Listening...'
    mic_btn.style.button_color = '#c62828'
    with mic_out:
        clear_output()
        display(HTML(
            '<div style="color:white;padding:8px;font-family:Poppins">'
            '🎤 Listening... speak now</div>'
        ))

    display(Javascript('''
    (() => {
      const SR = window.SpeechRecognition || window.webkitSpeechRecognition;
      if (!SR) { alert("Please use Chrome browser"); return; }
      navigator.mediaDevices.getUserMedia({ audio: true })
        .then(stream => {
          stream.getTracks().forEach(t => t.stop());
          let attempts = 0;
          function start() {
            if (attempts >= 3) return;
            attempts++;
            console.log("STT attempt", attempts);
            const r = new SR();
            r.lang = "en-ZA"; r.continuous = false; r.interimResults = false;
            r.onresult = (e) => {
              const t = e.results[0][0].transcript;
              console.log("Got transcript:", t);
              google.colab.kernel.invokeFunction("notebook.set_transcript", [t], {});
            };
            r.onerror = (e) => {
              console.log("STT error:", e.error);
              if (e.error === "network" || e.error === "aborted") setTimeout(start, 800);
              else google.colab.kernel.invokeFunction("notebook.set_transcript", [""], {});
            };
            r.onend = () => console.log("Listening ended");
            r.start();
          }
          start();
        })
        .catch(() => alert("Microphone access denied. Allow mic in Chrome settings."));
    })();
    '''))


mic_btn.on_click(on_mic_click)
builtins.mic_btn   = mic_btn
builtins.reset_mic = reset_mic

display(HTML(
    '<div style="display:flex;justify-content:center;'
    'padding:16px 0;font-family:Poppins,sans-serif">'
))
display(mic_btn)
display(HTML('</div>'))
display(mic_out)

with mic_out:
    display(HTML(
        '<div style="color:white;padding:8px;font-family:Poppins">'
        'Ready - tap to speak</div>'
    ))

print('Microphone ready!')

Button(description='🎤  Tap to Speak', layout=Layout(height='180px', width='180px'), style=ButtonStyle(button_c…

Output()

Microphone ready!


## Step 6 - Text Input Fallback

Use this if the microphone is not working.


In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

text_input = widgets.Text(
    placeholder='Type your message... e.g. Play Jerusalema',
    layout=widgets.Layout(width='100%')
)
submit_btn = widgets.Button(
    description='Submit',
    button_style='primary',
    layout=widgets.Layout(width='100%', margin='8px 0 0 0')
)
fallback_out = widgets.Output()


def on_submit(b):
    if text_input.value.strip():
        handle_voice_input(text_input.value.strip())
        text_input.value = ''


submit_btn.on_click(on_submit)

display(HTML(
    '<div style="font-family:Poppins,sans-serif;color:white;'
    'font-weight:700;margin-bottom:8px">Type a message:</div>'
))
display(text_input)
display(submit_btn)
display(fallback_out)
print('Text fallback ready!')

Text(value='', layout=Layout(width='100%'), placeholder='Type your message... e.g. Play Jerusalema')

Button(button_style='primary', description='Submit', layout=Layout(margin='8px 0 0 0', width='100%'), style=Bu…

Output()

Text fallback ready!


**<h3> Testing Accuracy Score for Frustration Detection Model </h3>**

True Positives (TP): The user sounded frustrated, and the app correctly detected it.

False Positives (FP): The user was calm, but the app triggered the "calming" response anyway.

True Negatives (TN): The user was calm, and the app stayed in its normal mode.

False Negatives (FN): The user was frustrated, but the app didn't notice and kept acting normally.




In [ ]:

tp =  16  # Correctly caught frustration
fp =  6 # False alarms
fn =  7  # Missed frustration
tn =  21 # Correctly identified calm

precision = tp / (tp + fp)
recall = tp / (tp + fn)
f1_score = 2 * (precision * recall) / (precision + recall)

print(f"Precision (Reliability): {precision:.2%}")
print(f"Recall (Sensitivity): {recall:.2%}")
print(f"F1-Score : {f1_score:.2%}")

Precision (Reliability): 72.73%
Recall (Sensitivity): 69.57%
F1-Score : 71.11%
